<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 02 — Silver: ICFES Saber 11

Limpieza y normalización del dataset de resultados Saber 11.

## FILTROS APLICADOS

| # | Filtro | Justificación |
|---|---|---|
| F1 | `PUNT_GLOBAL IS NOT NULL` | El puntaje global es la **variable objetivo del modelo supervisado**; un estudiante sin puntaje no aporta señal y sesga las agregaciones municipales. Elimina ~37% de filas (estudiantes que no presentaron). |
| F2 | `PUNT_GLOBAL > 0` | Puntaje cero indica registro incompleto/inválido (en Saber 11 el mínimo posible al presentar es ~50). Estos registros distorsionan los promedios municipales. |
| F3 | `PUNT_GLOBAL < 500` | El máximo teórico de Saber 11 es 500. Valores superiores son errores de captura o atípicos extremos no admisibles. |

## TRANSFORMACIONES APLICADAS

| # | Transformación | Justificación |
|---|---|---|
| T1 | Cast `PUNT_GLOBAL` → IntegerType | Bronze lo preserva como string; sin tipado no se pueden hacer agregaciones (avg, stddev) ni usar como label en MLlib. |
| T2 | Cast `PUNT_LECTURA_CRITICA`, `PUNT_MATEMATICAS`, `PUNT_C_NATURALES`, `PUNT_SOCIALES_CIUDADANAS`, `PUNT_INGLES` → DoubleType vía `regexp_replace(',', '.')` | Los puntajes por área pueden venir con coma decimal (locale es-CO, ej. `"55,2"`); el cast directo a numérico daría null. |
| T3 | Estandarizar `FAMI_TIENEINTERNET` y `FAMI_TIENECOMPUTADOR` → `UPPER` + `coalesce("SIN INFORMACION")` | Imputación explícita de nulos en categóricas críticas (en lugar de drop) preserva el registro para análisis ML donde el "no responde" es información en sí mismo. |
| T4 | Derivar binarias `TIENE_INTERNET_BIN`, `TIENE_COMPUTADOR_BIN` (Si→1, otro→0) | Requeridas para correlaciones de Pearson, promedios municipales (`pct_internet_icfes`) y features de modelos. |
| T5 | Normalizar nombres geográficos: `UPPER` + `translate` de acentos (`Á→A`, etc.) sobre `COLE_DEPTO_UBICACION` y `COLE_MCPIO_UBICACION` | Evita duplicación lógica por diferencias de capitalización/acento (`BOGOTÁ` vs `BOGOTA`). |
| T6 | `lpad(COLE_COD_DEPTO_UBICACION, 2, '0')` → `COD_DEPTO` ; `lpad(COLE_COD_MCPIO_UBICACION, 5, '0')` → `COD_MPIO` | Códigos DANE son claves de 2 y 5 dígitos. El zero-padding garantiza join correcto contra Internet/SISBEN/MEN. |
| T7 | Derivar `ANO = int(substring(PERIODO, 1, 4))` | El campo `PERIODO` es `YYYYS` (año + semestre, ej. `20194` = 2019-II). Extraer año permite particionar y comparar por cohorte. |
| T8 | Bucketizar `RANGO_PUNT_GLOBAL` con `when`: `<200`=BAJO, `<300`=MEDIO, resto=ALTO | Feature categórica derivada útil para clasificación y análisis exploratorio (Q5). |

## 1. Setup

In [1]:
import sys
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = get_spark("Entrega2-Silver-ICFES", executor_memory="4g", driver_memory="2g", cores=2)
print("App ID:", spark.sparkContext.applicationId)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/22 08:12:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


App ID: app-20260522081210-0034


## 2. Cargar Bronze ICFES

In [2]:
icfes_bronze = spark.read.parquet(P.BRONZE_PQ_ICFES)
print(f"Filas bronze: {icfes_bronze.count():,}")
print(f"Columnas    : {len(icfes_bronze.columns)}")
icfes_bronze.printSchema()

Filas bronze: 7,109,704
Columnas    : 51
root
 |-- PERIODO: string (nullable = true)
 |-- ESTU_TIPODOCUMENTO: string (nullable = true)
 |-- ESTU_CONSECUTIVO: string (nullable = true)
 |-- COLE_AREA_UBICACION: string (nullable = true)
 |-- COLE_BILINGUE: string (nullable = true)
 |-- COLE_CALENDARIO: string (nullable = true)
 |-- COLE_CARACTER: string (nullable = true)
 |-- COLE_COD_DANE_ESTABLECIMIENTO: string (nullable = true)
 |-- COLE_COD_DANE_SEDE: string (nullable = true)
 |-- COLE_COD_DEPTO_UBICACION: string (nullable = true)
 |-- COLE_COD_MCPIO_UBICACION: string (nullable = true)
 |-- COLE_CODIGO_ICFES: string (nullable = true)
 |-- COLE_DEPTO_UBICACION: string (nullable = true)
 |-- COLE_GENERO: string (nullable = true)
 |-- COLE_JORNADA: string (nullable = true)
 |-- COLE_MCPIO_UBICACION: string (nullable = true)
 |-- COLE_NATURALEZA: string (nullable = true)
 |-- COLE_NOMBRE_ESTABLECIMIENTO: string (nullable = true)
 |-- COLE_NOMBRE_SEDE: string (nullable = true)
 |-- COLE_SE

## 3. Filtrar registros sin puntaje global y atípicos

In [3]:
df = (
    icfes_bronze
    .withColumn("PUNT_GLOBAL", F.col("PUNT_GLOBAL").cast(IntegerType()))
    .filter(F.col("PUNT_GLOBAL").isNotNull())
    .filter(F.col("PUNT_GLOBAL") > 0)
    .filter(F.col("PUNT_GLOBAL") < 500)  # max teórico Saber 11 es 500
)
print(f"Filas tras filtro: {df.count():,}")

Filas tras filtro: 4,500,067


## 4. Tipificar puntajes por área

In [4]:
# Los puntajes por área pueden llegar como "55" (int) o "55,2" (decimal con coma — locale es-CO).
# Solución segura: cast→string, replace coma por punto, cast→double.
PUNT_COLS = [c for c in df.columns if c.startswith("PUNT_") and c != "PUNT_GLOBAL"]
print("Cols de puntaje por área:", PUNT_COLS)
for c in PUNT_COLS:
    df = df.withColumn(c,
                       F.regexp_replace(F.col(c).cast("string"), ",", ".").cast(DoubleType()))
df.select(["PUNT_GLOBAL"] + PUNT_COLS).describe().show()

Cols de puntaje por área: ['PUNT_INGLES', 'PUNT_MATEMATICAS', 'PUNT_SOCIALES_CIUDADANAS', 'PUNT_C_NATURALES', 'PUNT_LECTURA_CRITICA']


26/05/22 08:12:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+------------------------+------------------+--------------------+
|summary|       PUNT_GLOBAL|       PUNT_INGLES|  PUNT_MATEMATICAS|PUNT_SOCIALES_CIUDADANAS|  PUNT_C_NATURALES|PUNT_LECTURA_CRITICA|
+-------+------------------+------------------+------------------+------------------------+------------------+--------------------+
|  count|           4500067|           4495888|           4500067|                 4500067|           4500067|             4500067|
|   mean|252.30970849989566| 50.32080069610275|50.737230356792466|      48.866595764018626|50.107329513093916|  52.186261448996206|
| stddev| 50.41132948396743|12.534853891686534| 12.00887507754365|       11.72812584698333| 10.56670992277998|  10.382581653200534|
|    min|                 1|               0.0|               0.0|                     0.0|               0.0|                 0.0|
|    max|               495|             100.0|             100.0|          

## 5. Normalizar tecnología en el hogar

In [5]:
# Estandarizar mayúsculas y rellenar nulos
df = df.withColumn("FAMI_TIENEINTERNET",
                   F.upper(F.coalesce(F.col("FAMI_TIENEINTERNET"), F.lit("SIN INFORMACION"))))
df = df.withColumn("FAMI_TIENECOMPUTADOR",
                   F.upper(F.coalesce(F.col("FAMI_TIENECOMPUTADOR"), F.lit("SIN INFORMACION"))))

# Binarias (Si → 1, No/SIN INFORMACION → 0). En Saber 11 los valores son "Si"/"No" originalmente.
df = (df
    .withColumn("TIENE_INTERNET_BIN", F.when(F.col("FAMI_TIENEINTERNET") == "SI", 1).otherwise(0))
    .withColumn("TIENE_COMPUTADOR_BIN", F.when(F.col("FAMI_TIENECOMPUTADOR") == "SI", 1).otherwise(0))
)
df.groupBy("FAMI_TIENEINTERNET", "TIENE_INTERNET_BIN").count().orderBy(F.desc("count")).show(10)

+------------------+------------------+-------+
|FAMI_TIENEINTERNET|TIENE_INTERNET_BIN|  count|
+------------------+------------------+-------+
|                SI|                 1|2632130|
|                NO|                 0|1705658|
|   SIN INFORMACION|                 0| 162279|
+------------------+------------------+-------+



## 6. Normalizar nombres geográficos

In [6]:
# UPPER + remover acentos vía translate (ÁÉÍÓÚáéíóúÑñ → AEIOUaeiouNn)
ACCENTS_FROM = "ÁÉÍÓÚÜÑáéíóúüñ"
ACCENTS_TO   = "AEIOUUNaeiouun"

df = (df
    .withColumn("COLE_DEPTO_UBICACION", F.upper(F.translate(F.col("COLE_DEPTO_UBICACION"), ACCENTS_FROM, ACCENTS_TO)))
    .withColumn("COLE_MCPIO_UBICACION", F.upper(F.translate(F.col("COLE_MCPIO_UBICACION"), ACCENTS_FROM, ACCENTS_TO)))
)
# Rellenar códigos DANE a la longitud canónica
df = (df
    .withColumn("COD_DEPTO", F.lpad(F.col("COLE_COD_DEPTO_UBICACION"), 2, "0"))
    .withColumn("COD_MPIO",  F.lpad(F.col("COLE_COD_MCPIO_UBICACION"), 5, "0"))
)
df.select("COLE_DEPTO_UBICACION","COD_DEPTO","COLE_MCPIO_UBICACION","COD_MPIO").show(5, truncate=False)

+--------------------+---------+--------------------+--------+
|COLE_DEPTO_UBICACION|COD_DEPTO|COLE_MCPIO_UBICACION|COD_MPIO|
+--------------------+---------+--------------------+--------+
|HUILA               |41       |AIPE                |41016   |
|HUILA               |41       |AIPE                |41016   |
|HUILA               |41       |LA PLATA            |41396   |
|HUILA               |41       |LA PLATA            |41396   |
|VALLE               |76       |CALI                |76001   |
+--------------------+---------+--------------------+--------+
only showing top 5 rows



## 7. Derivar `ANO` desde PERIODO (YYYYS → YYYY)

In [7]:
df = df.withColumn("ANO", F.substring(F.col("PERIODO"), 1, 4).cast(IntegerType()))
df.groupBy("ANO").count().orderBy("ANO").show(30)

+----+-------+
| ANO|  count|
+----+-------+
|2014| 546637|
|2015| 570460|
|2016| 563369|
|2017| 561285|
|2018|  32348|
|2019|1109077|
|2020|  15434|
|2021|  15528|
|2022|1085929|
+----+-------+



## 8. Banding del puntaje global

In [8]:
df = df.withColumn(
    "RANGO_PUNT_GLOBAL",
    F.when(F.col("PUNT_GLOBAL") < 200, "BAJO")
     .when((F.col("PUNT_GLOBAL") >= 200) & (F.col("PUNT_GLOBAL") < 300), "MEDIO")
     .otherwise("ALTO")
)
df.groupBy("RANGO_PUNT_GLOBAL").count().orderBy(F.desc("count")).show()

+-----------------+-------+
|RANGO_PUNT_GLOBAL|  count|
+-----------------+-------+
|            MEDIO|2975241|
|             ALTO| 825691|
|             BAJO| 699135|
+-----------------+-------+



## 9. Selección de columnas finales y persistencia

In [9]:
# Conservamos columnas útiles para análisis Gold + features ML
COLS_FINAL = [
    "PERIODO", "ANO",
    "ESTU_CONSECUTIVO",
    "COD_DEPTO", "COD_MPIO",
    "COLE_DEPTO_UBICACION", "COLE_MCPIO_UBICACION",
    "COLE_AREA_UBICACION", "COLE_BILINGUE", "COLE_CALENDARIO", "COLE_CARACTER",
    "COLE_NATURALEZA", "COLE_JORNADA", "COLE_GENERO",
    "ESTU_GENERO",
    "FAMI_ESTRATOVIVIENDA",
    "FAMI_TIENEINTERNET", "FAMI_TIENECOMPUTADOR",
    "TIENE_INTERNET_BIN", "TIENE_COMPUTADOR_BIN",
    "PUNT_GLOBAL", "RANGO_PUNT_GLOBAL",
] + [c for c in df.columns if c.startswith("PUNT_") and c not in ("PUNT_GLOBAL",)]

COLS_FINAL = [c for c in COLS_FINAL if c in df.columns]
print("Cols guardadas:", len(COLS_FINAL))

df_silver = df.select(*COLS_FINAL)
df_silver.printSchema()

Cols guardadas: 27
root
 |-- PERIODO: string (nullable = true)
 |-- ANO: integer (nullable = true)
 |-- ESTU_CONSECUTIVO: string (nullable = true)
 |-- COD_DEPTO: string (nullable = true)
 |-- COD_MPIO: string (nullable = true)
 |-- COLE_DEPTO_UBICACION: string (nullable = true)
 |-- COLE_MCPIO_UBICACION: string (nullable = true)
 |-- COLE_AREA_UBICACION: string (nullable = true)
 |-- COLE_BILINGUE: string (nullable = true)
 |-- COLE_CALENDARIO: string (nullable = true)
 |-- COLE_CARACTER: string (nullable = true)
 |-- COLE_NATURALEZA: string (nullable = true)
 |-- COLE_JORNADA: string (nullable = true)
 |-- COLE_GENERO: string (nullable = true)
 |-- ESTU_GENERO: string (nullable = true)
 |-- FAMI_ESTRATOVIVIENDA: string (nullable = true)
 |-- FAMI_TIENEINTERNET: string (nullable = false)
 |-- FAMI_TIENECOMPUTADOR: string (nullable = false)
 |-- TIENE_INTERNET_BIN: integer (nullable = false)
 |-- TIENE_COMPUTADOR_BIN: integer (nullable = false)
 |-- PUNT_GLOBAL: integer (nullable = tru

## 10. Escribir Silver Parquet (particionado por ANO)

In [10]:
import time
t0 = time.time()
(df_silver.write
    .mode("overwrite")
    .partitionBy("ANO")
    .option("compression", "snappy")
    .parquet(P.SILVER_ICFES))
print(f"Escrito en {time.time()-t0:.1f}s")

Escrito en 34.5s


## 11. Verificación post-escritura

In [11]:
silver = spark.read.parquet(P.SILVER_ICFES)
n_final = silver.count()
print(f"Filas Silver: {n_final:,}")
print("Distribución por año:")
silver.groupBy("ANO").count().orderBy("ANO").show(30)
print("Promedio de puntaje por año:")
silver.groupBy("ANO").agg(F.round(F.avg("PUNT_GLOBAL"),1).alias("avg_punt"),
                          F.count("*").alias("n")).orderBy("ANO").show(30)

Filas Silver: 4,500,067
Distribución por año:


+----+-------+
| ANO|  count|
+----+-------+
|2014| 546637|
|2015| 570460|
|2016| 563369|
|2017| 561285|
|2018|  32348|
|2019|1109077|
|2020|  15434|
|2021|  15528|
|2022|1085929|
+----+-------+

Promedio de puntaje por año:


+----+--------+-------+
| ANO|avg_punt|      n|
+----+--------+-------+
|2014|   250.1| 546637|
|2015|   251.0| 570460|
|2016|   259.8| 563369|
|2017|   256.6| 561285|
|2018|   299.9|  32348|
|2019|   247.0|1109077|
|2020|   289.5|  15434|
|2021|   301.8|  15528|
|2022|   250.8|1085929|
+----+--------+-------+



In [12]:
spark.stop()